[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# sa_column and __table_args__


## What you will be able to do

Write a column `Field` has no argument for, with `sa_column`, and know the rule that comes with it:
what `sa_column` is given replaces what `Field` would have said rather than adding to it. Store a
dictionary or a list in one column as JSON, and read it back as a dictionary or a list. Put a
constraint over two columns, or a check the database enforces, in `__table_args__`. Give every
constraint a name with one convention, which the **Migrations** notebook depends on. Recognize the
failures:
the argument that may not travel with `sa_column`, a `Column` used twice, a default the table never
heard of, and the check that refuses a row.


## The idea

### The problem

`Field` covers what most columns need: a type from the annotation, a length, a default, a key, an
index, a foreign key. Past that it runs out, and it runs out in ways that are easy to meet.

A field holding a dictionary has no column type at all, and the class refuses to be defined, which
the **Field Types and Defaults** notebook met and handed forward to here. A rule that two columns
together must be unique cannot be written on either of them. A check the database itself enforces,
such as a number that may not be negative, has no `Field` argument. A column default that every
writer gets, rather than the model's own, needs the column to carry it. And on PostgreSQL there are
types with no Python equivalent to infer from, `JSONB` and `ARRAY` among them.

All of those are things SQLAlchemy has always been able to say. SQLModel's answer is not to grow an
argument for each of them: it is to let you write the SQLAlchemy `Column` yourself, and to let the
class carry SQLAlchemy's `__table_args__`. That is the seam, and it is worth knowing exactly how it
behaves, because what goes through it stops being SQLModel's business entirely.

### What sa_column and __table_args__ are

> **`Field(sa_column=Column(...))`** gives the field a column you wrote, with its type, its
> constraints, its default and anything else `Column` takes. It **replaces** what `Field` would have
> built, so `primary_key`, `index`, `unique`, `nullable` and `foreign_key` may not be passed beside
> it; they go inside the `Column`. **`__table_args__`**, a tuple on the class, carries what belongs
> to the table rather than to one column: **`UniqueConstraint`** over several columns,
> **`CheckConstraint`**, an `Index` over more than one column. A **naming convention** on
> `SQLModel.metadata` gives every constraint a name, which is what a migration tool needs to alter
> one later.

### Why it works that way

- **A column is one object.** `Field` builds one from its arguments; given one, it uses yours. Two
  descriptions of one column would have to be merged, and the rules for merging them would be
  another thing to learn, so SQLModel refuses instead.
- **`default` is Pydantic's, not the column's.** It survives beside `sa_column` because it never was
  a column argument: it is what the model fills in. The table's own default is `server_default`,
  inside the `Column`.
- **A `Column` belongs to one table.** Reusing an instance in a second class fails, because the
  first table already owns it.
- **Table constraints are not column constraints.** A rule about two columns has no single column to
  live on, so it goes in `__table_args__`.
- **A constraint with no name cannot be altered.** SQLite in particular needs names to rebuild a
  table, which the **Migrations** notebook needs and this notebook sets up.

### Where this shows up

JSON columns, which most applications end up with. Money and measurements that need a check.
Business rules that span two columns, such as one row per code per day. PostgreSQL's own types when
the same models are deployed there. The **SQLAlchemy, Deep Dive** guide's Tables and Metadata and
Column Types notebooks are where all of this is SQLAlchemy's to explain; this notebook is about
reaching it from SQLModel and about what that costs.

### What this notebook covers

- The field with no column type, and `sa_column`
- What `sa_column` replaces
- `default` beside `sa_column`, and the table's own default
- `__table_args__`: a rule over two columns, and a check
- A name for every constraint
- The same column on PostgreSQL
- A debrief written and read, finished
- Four failures, from an argument that may not travel to a check that refuses a row

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlalchemy import JSON, CheckConstraint, Column
from sqlmodel import Field, SQLModel


class Debrief(SQLModel, table=True):
    __table_args__ = (CheckConstraint("pages > 0", name="pages_are_positive"),)

    id: int | None = Field(default=None, primary_key=True)
    pages: int = Field(default=1)
    notes: dict = Field(default_factory=dict, sa_column=Column(JSON))


for column in Debrief.__table__.columns:
    print(f"{column.name:<7} {column.type}")
print("table rules:", sorted(rule.name for rule in Debrief.__table__.constraints if rule.name))
```

```
id      INTEGER
pages   INTEGER
notes   JSON
table rules: ['pages_are_positive']
```

The `dict` that had no column type has one, because the column was written rather than inferred. The
check is on the table rather than on a column, because it is about a column's values and belongs to
the table that holds them. It has a name because it was given one, and the primary key beside it has
none, which the last section of this notebook fixes for every table at once.


## Setup

Twelve imports, one of them installed first where it is missing, the cast, three helpers, a naming
convention, the classes, the engine, and the database built and loaded.

- `sqlmodel` is the library, and `SQLModel`, `Field`, `Relationship`, `Session`, `create_engine` and
  `select`, from it, are the classes, the session and the reading. Colab does not have SQLModel, so
  the cell installs 0.0.42 with `pip` where it is missing, and `version` and `PackageNotFoundError`,
  from `importlib.metadata`, `subprocess` and `sys` find out whether it is
- `Column`, `Integer`, `String` and `JSON`, from `sqlalchemy`, are what a column is written with when
  `Field` has no argument for it, and `UniqueConstraint` and `CheckConstraint` are what
  `__table_args__` carries
- `event`, `insert` and `text`, from `sqlalchemy`, are the pragma on every connection, the rows
  `build` loads without a session, and the two rows this notebook writes as plain SQL
- `IntegrityError`, from `sqlalchemy.exc`, is what a check refuses a row with
- `CreateTable`, from `sqlalchemy.schema`, with `sqlite` and `postgresql`, from
  `sqlalchemy.dialects`, write a table's SQL for this database and for one this notebook never
  connects to
- `re` takes memory addresses out of a message, `Path` names the database file, and `shutil` removes
  the scratch folder at the start and at the end
- `TEAMS` and `HEROES` are the cast, which `build` loads

The naming convention is set on `SQLModel.metadata` before any class is defined, because it applies
to the constraints of every table made afterwards. It is the last worked example's subject, and it
is here rather than there for that reason.


In [1]:
import re
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlmodel
from sqlalchemy import CheckConstraint, Column, Integer, JSON, String, UniqueConstraint, event, insert, text
from sqlalchemy.dialects import postgresql, sqlite
from sqlalchemy.exc import IntegrityError
from sqlalchemy.schema import CreateTable
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine, select

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


def table_sql(model):
    """The CREATE TABLE a table model describes, written for SQLite with no database anywhere."""
    return str(CreateTable(model.__table__).compile(dialect=sqlite.dialect())).strip()

SQLModel.metadata.naming_convention = {                             # every constraint gets a name from its shape
    "ix": "ix_%(table_name)s_%(column_0_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "pk": "pk_%(table_name)s",
}


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)

    heroes: list["Hero"] = Relationship(back_populates="team")


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")

    team: Team | None = Relationship(back_populates="heroes")


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


def build(engine):
    """Create the tables and load the cast, without a session: every notebook starts from the same rows."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS])
        teams = {name: number for number, (name, _) in enumerate(TEAMS, start=1)}
        connection.execute(insert(Hero), [{"name": name, "secret_name": secret, "age": age,
                                           "team_id": teams.get(team)}
                                          for name, secret, age, team in HEROES])

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same eight heroes
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build(engine)

with Session(engine) as session:
    print("sqlmodel", sqlmodel.__version__, "|", len(session.exec(select(Hero)).all()), "heroes |",
          "naming convention:", len(SQLModel.metadata.naming_convention), "rules")


sqlmodel 0.0.42 | 8 heroes | naming convention: 5 rules


## Worked examples

### The field with no column type, and sa_column

The **Field Types and Defaults** notebook met a field typed `dict` and the `ValueError` it raises,
and handed the answer here. This is it:


In [2]:
class Debrief(SQLModel, table=True):
    """What a mission left behind: a report, some notes, and a rule about both."""

    __table_args__ = (UniqueConstraint("mission_code", "written_on", name="one_a_day"),
                      CheckConstraint("pages > 0", name="pages_are_positive"))

    id: int | None = Field(default=None, primary_key=True)
    mission_code: str = Field(max_length=20)
    written_on: str = Field(max_length=10)                          # a date as text, to keep the key readable
    pages: int = Field(default=1)
    notes: dict = Field(default_factory=dict, sa_column=Column(JSON))
    tags: list[str] = Field(default_factory=list, sa_column=Column(JSON))
    filed_by: str | None = Field(default=None,
                                 sa_column=Column(String(40), server_default=text("'the registry'")))


SQLModel.metadata.create_all(engine)
print(table_sql(Debrief))


CREATE TABLE debrief (
	id INTEGER NOT NULL, 
	mission_code VARCHAR(20) NOT NULL, 
	written_on VARCHAR(10) NOT NULL, 
	pages INTEGER NOT NULL, 
	notes JSON, 
	tags JSON, 
	filed_by VARCHAR(40) DEFAULT 'the registry', 
	CONSTRAINT pk_debrief PRIMARY KEY (id), 
	CONSTRAINT one_a_day UNIQUE (mission_code, written_on), 
	CONSTRAINT ck_debrief_pages_are_positive CHECK (pages > 0)
)


`notes` and `tags` are `JSON` columns, written as `Column(JSON)` because no annotation implies them.
SQLAlchemy stores the value as JSON text on SQLite and reads it back as Python, which the round trip
below shows. `filed_by` is a `String(40)` with a default the table carries, and the two constraints
in `__table_args__` are in the `CREATE TABLE` with the names they were given.

What goes in a JSON column is a value, not a query. Asking the database for every debrief whose
notes mention the bridge means a function that database has for looking inside JSON, and those
differ between databases; a column of its own, or a table, is what to write when a value is searched
rather than stored.


In [3]:
with Session(engine) as session:
    session.add(Debrief(mission_code="BRIDGE-1", written_on="2026-03-01", pages=4,
                        notes={"weather": "rain", "hours": 3}, tags=["night", "rescue"]))
    session.commit()

with Session(engine) as session:
    written = session.exec(select(Debrief)).one()
    print("notes:", written.notes, type(written.notes).__name__)
    print("tags :", written.tags, type(written.tags).__name__)
    print("filed by:", written.filed_by)


notes: {'weather': 'rain', 'hours': 3} dict
tags : ['night', 'rescue'] list
filed by: the registry


A dictionary in and a dictionary out, a list in and a list out, and `filed_by` filled in by the
table, since the model never set it.

### What sa_column replaces

The rule is one sentence: what `sa_column` is given is the whole column. Anything `Field` would have
put in it has to be in it:


In [4]:
class Sighting(SQLModel, table=True):
    id: int | None = Field(default=None, sa_column=Column(Integer, primary_key=True))
    city: str = Field(sa_column=Column(String(60), index=True, nullable=False, unique=True))


SQLModel.metadata.create_all(engine)
print(table_sql(Sighting))
print("indexes:", [(index.name, "unique" if index.unique else "not unique")
                   for index in Sighting.__table__.indexes])


CREATE TABLE sighting (
	id INTEGER NOT NULL, 
	city VARCHAR(60) NOT NULL, 
	CONSTRAINT pk_sighting PRIMARY KEY (id)
)
indexes: [('ix_sighting_city', 'unique')]


`primary_key`, `index`, `nullable` and `unique` are all there, and all of them inside the `Column`.
Passing any of them to `Field` beside `sa_column` raises, which is the first of the Common errors,
and the reason is worth keeping: `Field` builds a column from its arguments, and given one it uses
what it was given rather than merging two descriptions.

The `CREATE TABLE` has no `UNIQUE` in it, and the city is unique all the same: asking for an index
and uniqueness on one column makes one unique index, which is what the second line prints. Its name
came from the convention, which the last section is about.

### default beside sa_column, and the table's own default

One argument does survive next to `sa_column`, and it is the one that was never a column argument:


In [5]:
class Report(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    pages: int = Field(default=3, sa_column=Column(Integer))        # default is Pydantic's, not the column's


SQLModel.metadata.create_all(engine)
print("the model fills in:", Report().pages)
print("the column's default:", Report.__table__.columns["pages"].default)
print(table_sql(Report))

with engine.begin() as connection:
    connection.execute(text("INSERT INTO report (pages) VALUES (NULL)"))    # a writer that is not the model
with Session(engine) as session:
    print("a row written past the model:", session.exec(select(Report)).one().pages)


the model fills in: 3
the column's default: None
CREATE TABLE report (
	id INTEGER NOT NULL, 
	pages INTEGER, 
	CONSTRAINT pk_report PRIMARY KEY (id)
)
a row written past the model: None


`default=3` is what a `Report()` gets, and the table knows nothing about it: the `CREATE TABLE` has
no `DEFAULT`, and a row written any other way has a null where the model would have put a 3. That is
the same distinction the **Field Types and Defaults** notebook drew, and `sa_column` is where it
becomes easy to get wrong, because the two defaults now sit on the same line.

`Column(..., server_default=...)` is the table's, as `Debrief.filed_by` has above.

### __table_args__: a rule over two columns, and a check

Neither of the `Debrief` rules could have been written on a field. One is about two columns
together, and the other is a condition the database checks on every write:


In [6]:
with Session(engine) as session:
    session.add(Debrief(mission_code="BRIDGE-1", written_on="2026-03-01", pages=2))
    try:
        session.commit()
    except IntegrityError as error:
        print("two on one day:", str(error).splitlines()[0])
    session.rollback()

    session.add(Debrief(mission_code="VAULT-9", written_on="2026-03-02", pages=0))
    try:
        session.commit()
    except IntegrityError as error:
        print("no pages      :", str(error).splitlines()[0])
    session.rollback()

    session.add(Debrief(mission_code="VAULT-9", written_on="2026-03-02", pages=2))
    session.commit()
    print("accepted      :", len(session.exec(select(Debrief)).all()), "debriefs")


two on one day: (sqlite3.IntegrityError) UNIQUE constraint failed: debrief.mission_code, debrief.written_on
no pages      : (sqlite3.IntegrityError) CHECK constraint failed: ck_debrief_pages_are_positive
accepted      : 2 debriefs


Both refusals name the constraint that made them, because both constraints have names. A rule with
no name is refused just the same, and the message says only that a constraint failed, which is a
poor thing to read in a log.

These are the database's rules rather than the model's, and that is the point of putting them there:
they hold for a migration, for another service, and for a line of SQL typed at two in the morning.
The model's own rules, the ones a `field_validator` writes, hold only for values that go through
`model_validate`, as the **Validation and table=True** notebook showed.

### A name for every constraint

The convention in Setup names every constraint from its shape, so nothing depends on remembering to
name one:


In [7]:
for model in (Hero, Debrief, Sighting):
    named = sorted(constraint.name for constraint in model.__table__.constraints if constraint.name)
    print(f"  {model.__name__:<9} {named}")
print("hero's foreign key:", [key.constraint.name for key in Hero.__table__.foreign_keys])


  Hero      ['fk_hero_team_id_team', 'pk_hero']
  Debrief   ['ck_debrief_pages_are_positive', 'one_a_day', 'pk_debrief']
  Sighting  ['pk_sighting']
hero's foreign key: ['fk_hero_team_id_team']


`pk_hero`, `pk_debrief`, `fk_hero_team_id_team` and `ix_sighting_city` were all named by the
convention, and nobody wrote any of them. The two rules that were named by hand went two different
ways: `one_a_day` is exactly what it was given, and the check came out
`ck_debrief_pages_are_positive`, because the `ck` rule in the convention has `%(constraint_name)s`
in it and wraps the name it was given. A rule whose pattern does not mention that is left alone when
it already has a name.

This matters for the **Migrations** notebook and not much before it. Changing or dropping a
constraint means
naming it, and SQLite has to rebuild the whole table to do either, so a migration tool needs a name
for everything. A project that sets the convention on its first day never has to find out what its
constraints are called.

### The same column on PostgreSQL

`sa_column` is also where a type that only one database has goes. Nothing here connects to
PostgreSQL; the dialect writes the SQL:


In [8]:
for dialect in (sqlite.dialect(), postgresql.dialect()):
    written = str(CreateTable(Debrief.__table__).compile(dialect=dialect)).strip()
    print(f"-- {dialect.name}")
    print("\n".join(line for line in written.splitlines() if "notes" in line or "tags" in line))


-- sqlite
	notes JSON, 
	tags JSON, 
-- postgresql
	notes JSON, 
	tags JSON, 


The same `JSON` column is `JSON` on both, which is as far as a portable type goes. PostgreSQL's own
`JSONB`, which is stored in a form it can index and search, is
`Column(postgresql.JSONB)`, and `ARRAY` is the same shape; both are honest reasons to reach for
`sa_column`, and both make the model that uses them a PostgreSQL model. The
**SQLAlchemy, Deep Dive** guide's Four Databases, One Codebase notebook is where that trade is taken
apart.

### A debrief written and read, finished

The pieces of this notebook in one function. `file_debrief` writes a debrief and reports what the
database refused, naming the rule that refused it:


In [9]:
def file_debrief(engine, **values):
    """Write one debrief, and name the rule that refused it where one did."""
    with Session(engine, expire_on_commit=False) as session:
        session.add(Debrief(**values))
        try:
            session.commit()
        except IntegrityError as error:
            session.rollback()
            first = str(error).splitlines()[0]
            return {"refused": first.split(": ")[-1]}
    return {"filed": values["mission_code"], "tags": values["tags"], "by": "the table's default"}


print(file_debrief(engine, mission_code="TOWER-3", written_on="2026-03-03", pages=2,
                   notes={"weather": "clear"}, tags=["day"]))
print(file_debrief(engine, mission_code="TOWER-3", written_on="2026-03-03", pages=2,
                   notes={}, tags=[]))
print(file_debrief(engine, mission_code="TOWER-4", written_on="2026-03-04", pages=0,
                   notes={}, tags=[]))


{'filed': 'TOWER-3', 'tags': ['day'], 'by': "the table's default"}
{'refused': 'debrief.mission_code, debrief.written_on'}
{'refused': 'ck_debrief_pages_are_positive'}


One filed, one refused by the rule about two columns, one refused by the check, and each refusal
names the rule rather than the column. The tags went in as a list and the `filed_by` came from the
table, neither of which `Field` alone could have arranged.

### Where each part came from

| In `file_debrief` and `Debrief` | What it relies on | The section that showed it |
|---|---|---|
| `Column(JSON)` on `notes` and `tags` | a column written rather than inferred | The field with no column type |
| `Column(String(40), server_default=...)` | the table's own default, inside the column | `default` beside `sa_column` |
| `UniqueConstraint("mission_code", "written_on", ...)` | a rule over two columns | `__table_args__` |
| `CheckConstraint("pages > 0", ...)` | a condition the database enforces on every write | `__table_args__` |
| the rule's name in the message | constraints that have names | A name for every constraint |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/11-sa-column-and-table-args-solutions.ipynb).

**1.** Write a `Gadget` table model with a `name`, and a `settings` field holding a dictionary, and
print its `CREATE TABLE`.


In [10]:
# your code here


**2.** Write a gadget whose settings are `{"mode": "stealth", "charges": 2}` and read it back,
printing the value and its type.


In [11]:
# your code here


**3.** Give `Gadget` a `serial` column that is unique and indexed, written entirely inside a
`Column`, and print the table and its indexes.


In [12]:
# your code here


**4.** Add a check that a gadget's `charges` may not be negative, and show it refusing a row and
accepting another.


In [13]:
# your code here


**5.** Give `Gadget` a `checked_by` column whose default is the table's, write a row with plain SQL
that does not name it, and print what came back.


In [14]:
# your code here


**6.** Print the names of every constraint on `Gadget`, and say in a comment which came from the
convention and which were given by hand.


In [15]:
# your code here


## Common errors

### RuntimeError: Passing primary_key is not supported when also passing a sa_column


In [16]:
class Badge(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True, sa_column=Column(Integer))
    name: str = Field(max_length=40)


RuntimeError: Passing primary_key is not supported when also passing a sa_column

The same message comes for `index`, `unique`, `nullable` and `foreign_key`: `sa_column` is the whole
column, and those all belong inside it. The fix is to move the argument rather than to remove it:


In [17]:
class Badge(SQLModel, table=True):
    id: int | None = Field(default=None, sa_column=Column(Integer, primary_key=True))
    name: str = Field(max_length=40)


print([f"{column.name} {column.type} key={column.primary_key}" for column in Badge.__table__.columns])


['id INTEGER key=True', 'name VARCHAR(40) key=False']


### sqlalchemy.exc.ArgumentError: Column object 'label' already assigned to Table 'firstplaque'


In [18]:
SHARED = Column(String(40))                                         # one object, about to be used twice


class FirstPlaque(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    label: str = Field(sa_column=SHARED)


class SecondPlaque(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    label: str = Field(sa_column=SHARED)


ArgumentError: Column object 'label' already assigned to Table 'firstplaque'

A `Column` is not a description that can be copied: it is an object that belongs to a table once it
is used. The same happens with a base class that has `sa_column` on a field and two models
inheriting it, which is the shape this usually arrives in.

Give each class its own, from a function when they should stay identical:


In [19]:
def label_column():
    """A new Column each time, so that two tables never share one."""
    return Column(String(40), nullable=False)


class SecondPlaque(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    label: str = Field(sa_column=label_column())


SQLModel.metadata.create_all(engine)
print("both tables:", [name for name in sorted(SQLModel.metadata.tables) if "plaque" in name])


both tables: ['firstplaque', 'secondplaque']


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sqlmodel/main.py:722: SAWarning: This declarative base already contains a class with the same class name and module name as __main__.SecondPlaque, and will be replaced in the string-lookup table.
  DeclarativeMeta.__init__(cls, classname, bases, dict_, **kw)


### No error, and a default the table never heard of


In [20]:
with engine.begin() as connection:
    connection.execute(text("INSERT INTO report (pages) VALUES (NULL)"))

with Session(engine) as session:
    print("pages of every report:", [report.pages for report in session.exec(select(Report))])


pages of every report: [None, None]


`Report.pages` has `default=3` and its column has no default at all, so a row written by anything
that is not the model has a null in it, and the model reads that null back as `None` in a field
annotated `int`. Nothing raised at any point.

Which default is wanted is the question to ask, and `sa_column` makes it easy to answer both ways:
`Field(default=3)` for the model, `Column(Integer, server_default=text("3"))` for the table, and
both where rows arrive from both.

### sqlalchemy.exc.IntegrityError: (sqlite3.IntegrityError) CHECK constraint failed: pages_are_positive


In [21]:
with Session(engine) as session:
    session.add(Debrief(mission_code="DOCKS-2", written_on="2026-03-05", pages=-1))
    session.commit()


IntegrityError: (sqlite3.IntegrityError) CHECK constraint failed: ck_debrief_pages_are_positive
[SQL: INSERT INTO debrief (mission_code, written_on, pages, notes, tags) VALUES (?, ?, ?, ?, ?) RETURNING id, filed_by]
[parameters: ('DOCKS-2', '2026-03-05', -1, '{}', '[]')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

The check did what it was written to do. It is in the Common errors because of what it means for the
code around it: a constraint in the database is not a substitute for checking at the edge, it is the
thing that catches what the edge missed, and a service that does not catch `IntegrityError` turns it
into a 500.

The pair to write is both: a rule in the model, so a caller is told which field is wrong, and a
constraint in the table, so nothing else can write the row. The **Validation and table=True**
notebook has the first half, and the **SQLModel in FastAPI** notebook turns the second into an
answer a client can read.

Last, the engine lets go of the file, and this cell removes the scratch folder with the database in
it:


In [22]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `Field(sa_column=Column(...))` is the seam to SQLAlchemy: the column you write is the column, and
  `primary_key`, `index`, `unique`, `nullable` and `foreign_key` go inside it rather than beside it.
- `default` survives next to `sa_column` because it is the model's, and `server_default` inside the
  `Column` is the table's.
- A `Column` object belongs to one table; two models need two of them.
- `__table_args__` carries what belongs to the table: a `UniqueConstraint` over two columns, a
  `CheckConstraint`, an index over more than one column.
- A naming convention on `SQLModel.metadata` names every constraint, which is what makes a message
  readable and a migration possible.


## What is next

The **Migrations** notebook takes a change to these classes and turns it into a change to a database
that already has rows in it: Alembic's autogenerate against `SQLModel.metadata`, the import it
leaves out of the script it writes, and the column `create_all` will never add.


---

&#8592; **Previous:** [Loading and N+1](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/10-loading-and-n-plus-one.ipynb)  &nbsp;·&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
